<a href="https://colab.research.google.com/github/bcbutler2212/Education-Policy-Research-Chatbot/blob/Sofia/Working_Llama_3_2_1B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U transformers

## Local Inference on GPU
Model page: https://huggingface.co/meta-llama/Llama-3.2-1B

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/meta-llama/Llama-3.2-1B)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

The model you are trying to use is gated. Please make sure you have access to it by visiting the model page.To run inference, either set HF_TOKEN in your environment variables/ Secrets or run the following cell to login. 🤗

In [ ]:
from huggingface_hub import login
login(new_session=False)

In [ ]:
# Use a pipeline as a high-level helper
from transformers import pipeline
import os
from google.colab import userdata

# Set the HF_TOKEN environment variable
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

pipe = pipeline("text-generation", model="meta-llama/Llama-3.2-1B")

In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B")
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-1B")

In [ ]:
# load files temporarily
from google.colab import files
uploaded = files.upload()

In [ ]:
# unzip
!unzip /content/pdfs_for_chatbot.zip

In [ ]:
# Install dependencies
!pip install langchain langchain-community pypdf chromadb sentence-transformers transformers

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [ ]:
import os
os.listdir("/content")

In [ ]:
# Load all pdfs

pdf_folder = "/content/pdfs_for_chatbot"

docs = []
for file in os.listdir(pdf_folder):
    if file.endswith(".pdf"):
        loader = PyPDFLoader(os.path.join(pdf_folder, file))
        docs.extend(loader.load())

print(f"Loaded {len(docs)} PDF pages")

In [ ]:

# Split long text into smaller chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
split_docs = splitter.split_documents(docs)

print(f"Created {len(split_docs)} text chunks")

In [ ]:
# wrap
from langchain_community.llms import HuggingFacePipeline
llm = HuggingFacePipeline(pipeline=pipe)

In [ ]:
# embedding model
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
# Create a persistent Chroma DB
vectordb = Chroma.from_documents(split_docs, embedding=embedding_model, persist_directory="chroma_db")


In [ ]:
vectordb.persist()

In [ ]:
# If chunks print then working

results = vectordb.similarity_search("What is this document about?", k=2)
for i, r in enumerate(results, 1):
    print(f"\nResult {i}:\n{r.page_content[:500]}...")

In [ ]:
# make retriever --> return 3 chunks? change this?
retriever = vectordb.as_retriever(search_kwargs={"k": 3})

In [ ]:
from langchain.chains import RetrievalQA

# connect llama with chroma
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff"
)

In [ ]:
# test it!
result = qa_chain.invoke({"query": "Summarize the key findings from the PDFs"})
print(result["result"])

In [21]:
question = "What are school distruptions?"
result = qa_chain.invoke({"query": question})
print(result["result"])

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

about school funding, concerns with financing mechanisms, or the role and impact of the 
courts. Nonetheless, the continuing application to the courts for changes in the structure 
of state school finance arrangements indicate s a degree of ongoing dissatisfaction with 
school policymaking.

school finance policy and decisions. 
Decision makers in the executive and legislative branches of state and federal 
governments, along with those in local school districts , have developed a variety of 
approaches for ensuring that schools have the resources necessary to do their job. The 
funding patterns that have resulted continue to be questioned on the grounds of 
inequitable opportunities across districts. Additionally, the results, as measured in terms 
of student outcomes , have not constituted the outcomes that are desired. 43